In [ ]:
%pip -q install datasets pandas matplotlib pyarrow sentence-transformers faiss-cpu requests rank_bm25

In [ ]:
import re
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

SEED = 42
np.random.seed(SEED)

print("All imports successful!")

In [ ]:
DATASET_ID = "allenai/qasper"

response = requests.get(
    "https://datasets-server.huggingface.co/parquet",
    params={"dataset": DATASET_ID},
    timeout=60,
)

response.raise_for_status()
parquet_info = response.json()

print("Available splits and files:")

for item in parquet_info["parquet_files"]:
    print(
        item["config"],
        item["split"],
        item["url"],
    )

In [ ]:
parquet_files = parquet_info["parquet_files"]

# Find the config containing the standard QASPER splits.
configs = list(dict.fromkeys(
    item["config"] for item in parquet_files
))

print("Available configs:", configs)

preferred_config = (
    "qasper" if "qasper" in configs else configs[0]
)

data_files = {}

for split_name in ["train", "validation", "test"]:
    urls = [
        item["url"]
        for item in parquet_files
        if item["config"] == preferred_config
        and item["split"] == split_name
    ]

    if urls:
        data_files[split_name] = urls

if not data_files:
    raise ValueError(
        "No usable Parquet splits found. Inspect parquet_info above."
    )

dataset = load_dataset(
    "parquet",
    data_files=data_files,
)

print(dataset)

In [ ]:
EVAL_SPLIT = "validation"

# Increase to None when you want to evaluate the entire split.
MAX_PAPERS = 30
MAX_QUESTIONS_PER_PAPER = 10

TOP_K = 5

CHUNK_SIZE = 200
CHUNK_OVERLAP = 40

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Split:", EVAL_SPLIT)
print("Max papers:", MAX_PAPERS)
print("Top K:", TOP_K)
print("Chunk size:", CHUNK_SIZE)
print("Overlap:", CHUNK_OVERLAP)

In [ ]:
split = dataset[EVAL_SPLIT]
paper = split[0]

print("Dataset size:", len(split))
print("Paper ID:", paper["id"])
print("Title:", paper["title"])

print("\nSections:")
for section in paper["full_text"]["section_name"][:5]:
    print("-", section)

print("\nFirst question:")
print(paper["qas"]["question"][0])

In [ ]:
n_papers = min(
    MAX_PAPERS if MAX_PAPERS is not None else len(split),
    len(split),
)

eval_papers = [split[i] for i in range(n_papers)]

print("Selected papers:", len(eval_papers))

In [ ]:
def extract_evidence(paper, question_index):
    record = paper["qas"]["answers"][question_index]
    evidence_list = []

    for annotation in record["answer"]:
        if annotation["unanswerable"]:
            continue

        for evidence in annotation["evidence"]:
            if not evidence or not evidence.strip():
                continue

            if evidence.strip().upper().startswith("FLOAT SELECTED"):
                continue

            evidence_list.append(evidence.strip())

    return list(dict.fromkeys(evidence_list))

In [ ]:
evaluation_questions = []

for paper in eval_papers:
    qas = paper["qas"]

    n_questions = len(qas["question"])

    if MAX_QUESTIONS_PER_PAPER is not None:
        n_questions = min(
            n_questions,
            MAX_QUESTIONS_PER_PAPER,
        )

    for i in range(n_questions):
        evidence = extract_evidence(paper, i)

        if not evidence:
            continue

        evaluation_questions.append({
            "paper_id": paper["id"],
            "title": paper["title"],
            "question_id": qas["question_id"][i],
            "question": qas["question"][i],
            "evidence": evidence,
        })

questions_df = pd.DataFrame(evaluation_questions)

print("Questions with textual evidence:", len(questions_df))
display(questions_df.head(3))

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

## Chunking experiments: Paragraph vs.Fixed Size vs. Section Aware

## Metrics: 
- recall_at_k
- mrr
- precision_at_k



In [ ]:
def paragraph_chunking(paper):
    chunks = []
    full_text = paper["full_text"]

    for section_idx, section_name in enumerate(
        full_text["section_name"]
    ):
        paragraphs = full_text["paragraphs"][section_idx]

        for paragraph_idx, paragraph in enumerate(paragraphs):
            if not paragraph or not paragraph.strip():
                continue

            chunks.append({
                "paper_id": paper["id"],
                "title": paper["title"],
                "section": section_name,
                "chunk_index": paragraph_idx,
                "text": paragraph.strip(),
            })

    return chunks

In [ ]:
def split_into_chunks(text, chunk_size=200, overlap=40):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError(
            "overlap must be >= 0 and smaller than chunk_size"
        )

    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]

        if not chunk_words:
            break

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

In [ ]:
def fixed_size_chunking(
    paper,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
):
    chunks = []
    full_text = paper["full_text"]

    for section_idx, section_name in enumerate(
        full_text["section_name"]
    ):
        paragraphs = full_text["paragraphs"][section_idx]

        for paragraph_idx, paragraph in enumerate(paragraphs):
            if not paragraph or not paragraph.strip():
                continue

            parts = split_into_chunks(
                paragraph.strip(),
                chunk_size=chunk_size,
                overlap=overlap,
            )

            for chunk_idx, text in enumerate(parts):
                chunks.append({
                    "paper_id": paper["id"],
                    "title": paper["title"],
                    "section": section_name,
                    "chunk_index": chunk_idx,
                    "paragraph_index": paragraph_idx,
                    "text": text,
                })

    return chunks

In [ ]:
def section_aware_chunking(
    paper,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
):
    chunks = []
    full_text = paper["full_text"]

    for section_idx, section_name in enumerate(
        full_text["section_name"]
    ):
        paragraphs = full_text["paragraphs"][section_idx]

        section_text = "\n\n".join(
            p.strip()
            for p in paragraphs
            if p and p.strip()
        )

        if not section_text:
            continue

        parts = split_into_chunks(
            section_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_idx, text in enumerate(parts):
            chunks.append({
                "paper_id": paper["id"],
                "title": paper["title"],
                "section": section_name,
                "chunk_index": chunk_idx,
                "text": text,
            })

    return chunks

In [ ]:
chunkers = {
    "paragraph": paragraph_chunking,
    "fixed_size": fixed_size_chunking,
    "section_aware": section_aware_chunking,
}

chunks_by_strategy = {}

for strategy, chunker in chunkers.items():
    all_chunks = []

    for paper in eval_papers:
        all_chunks.extend(chunker(paper))

    chunks_by_strategy[strategy] = all_chunks

    print(f"{strategy}: {len(all_chunks):,} chunks")

In [ ]:
chunk_stats = []

for strategy, chunks in chunks_by_strategy.items():
    lengths = [
        len(chunk["text"].split())
        for chunk in chunks
    ]

    chunk_stats.append({
        "strategy": strategy,
        "num_chunks": len(chunks),
        "mean_words": np.mean(lengths) if lengths else 0,
        "median_words": np.median(lengths) if lengths else 0,
        "max_words": max(lengths, default=0),
    })

chunk_stats_df = pd.DataFrame(chunk_stats)

display(chunk_stats_df)

chunk_stats_df.plot(
    x="strategy",
    y="num_chunks",
    kind="bar",
    legend=False,
    title="Number of chunks by strategy",
    ylabel="Number of chunks",
    xlabel="Chunking strategy",
    rot=0,
)

plt.tight_layout()
plt.show()

In [ ]:
model = SentenceTransformer(MODEL_NAME)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

In [ ]:
def build_paper_indexes(chunks, model):
    grouped = defaultdict(list)

    for chunk in chunks:
        grouped[chunk["paper_id"]].append(chunk)

    indexes = {}

    for paper_id, paper_chunks in grouped.items():
        texts = [
            (
                f"Title: {chunk['title']}\n"
                f"Section: {chunk['section']}\n"
                f"Content: {chunk['text']}"
            )
            for chunk in paper_chunks
        ]

        embeddings = model.encode(
            texts,
            batch_size=32,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype("float32")

        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

        indexes[paper_id] = {
            "index": index,
            "chunks": paper_chunks,
        }

    return indexes

In [ ]:
indexes_by_strategy = {}

for strategy, chunks in chunks_by_strategy.items():
    print("Building:", strategy)

    indexes_by_strategy[strategy] = build_paper_indexes(
        chunks,
        model,
    )

print("All indexes are ready.")

In [ ]:
def retrieve_chunks(
    question,
    paper_id,
    paper_indexes,
    top_k=TOP_K,
):
    paper_data = paper_indexes.get(paper_id)

    if paper_data is None:
        return []

    index = paper_data["index"]
    chunks = paper_data["chunks"]

    if index.ntotal == 0:
        return []

    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        min(top_k, index.ntotal),
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        results.append({
            **chunks[int(idx)],
            "score": float(score),
        })

    return results

In [ ]:
def chunk_matches_evidence(chunk_text, evidence_list):
    chunk_norm = normalize_text(chunk_text)

    if len(chunk_norm) < 80:
        return False

    for evidence in evidence_list:
        evidence_norm = normalize_text(evidence)

        if not evidence_norm:
            continue

        if evidence_norm in chunk_norm:
            return True

        if len(chunk_norm) >= 80 and chunk_norm in evidence_norm:
            return True

    return False

In [ ]:
def evaluate_strategy(strategy, paper_indexes):
    rows = []

    for q in evaluation_questions:
        paper_data = paper_indexes.get(q["paper_id"])

        if paper_data is None:
            continue

        all_chunks = paper_data["chunks"]

        # Identify relevant chunks throughout this paper's corpus.
        relevant_all = [
            chunk_matches_evidence(chunk["text"], q["evidence"])
            for chunk in all_chunks
        ]

        total_relevant = sum(relevant_all)

        if total_relevant == 0:
            rows.append({
                "strategy": strategy,
                "paper_id": q["paper_id"],
                "question_id": q["question_id"],
                "evaluated": False,
                "recall_at_k": np.nan,
                "mrr": np.nan,
                "precision_at_k": np.nan,
            })
            continue

        retrieved = retrieve_chunks(
            q["question"],
            q["paper_id"],
            paper_indexes,
            TOP_K,
        )

        relevance = [
            chunk_matches_evidence(
                item["text"],
                q["evidence"],
            )
            for item in retrieved
        ]

        relevant_ranks = [
            rank
            for rank, relevant in enumerate(relevance, start=1)
            if relevant
        ]

        recall = sum(relevance) / total_relevant

        mrr = (
            1.0 / relevant_ranks[0]
            if relevant_ranks else 0.0
        )

        precision = sum(relevance) / TOP_K

        rows.append({
            "strategy": strategy,
            "paper_id": q["paper_id"],
            "question_id": q["question_id"],
            "evaluated": True,
            "recall_at_k": recall,
            "mrr": mrr,
            "precision_at_k": precision,
        })

    return pd.DataFrame(rows)

In [ ]:
result_frames = []

for strategy, indexes in indexes_by_strategy.items():
    print("Evaluating:", strategy)

    result_frames.append(
        evaluate_strategy(strategy, indexes)
    )

results_df = pd.concat(
    result_frames,
    ignore_index=True,
)

print("Evaluation finished.")
display(results_df.head())

In [ ]:
summary_df = (
    results_df[results_df["evaluated"]]
    .groupby("strategy")
    .agg(
        evaluated_questions=("question_id", "count"),
        recall_at_k=("recall_at_k", "mean"),
        mrr=("mrr", "mean"),
        precision_at_k=("precision_at_k", "mean"),
    )
    .reset_index()
    .sort_values("recall_at_k", ascending=False)
)

display(summary_df.round(4))

In [ ]:
metrics = [
    "recall_at_k",
    "mrr",
    "precision_at_k",
]

summary_df.set_index("strategy")[metrics].plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1),
    title=f"Chunking strategy comparison (top-{TOP_K})",
)

plt.ylabel("Score")
plt.xlabel("Strategy")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
example = evaluation_questions[0]

print("Paper:", example["title"])
print("Question:", example["question"])

print("\nAnnotated evidence:")
for evidence in example["evidence"]:
    print("-", evidence[:400])

In [ ]:
for strategy, indexes in indexes_by_strategy.items():
    print("\n" + "=" * 80)
    print("STRATEGY:", strategy)

    retrieved = retrieve_chunks(
        example["question"],
        example["paper_id"],
        indexes,
        TOP_K,
    )

    for rank, chunk in enumerate(retrieved, start=1):
        relevant = chunk_matches_evidence(
            chunk["text"],
            example["evidence"],
        )

        print(
            f"\nRank {rank} | "
            f"Similarity: {chunk['score']:.3f} | "
            f"Evidence match: {relevant}"
        )
        print("Section:", chunk["section"])
        print(chunk["text"][:600])

In [ ]:
summary_df.to_csv("chunking_summary.csv", index=False)
results_df.to_csv("chunking_question_results.csv", index=False)
chunk_stats_df.to_csv("chunking_statistics.csv", index=False)

config = {
    "dataset": DATASET_ID,
    "split": EVAL_SPLIT,
    "max_papers": MAX_PAPERS,
    "max_questions_per_paper": MAX_QUESTIONS_PER_PAPER,
    "embedding_model": MODEL_NAME,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "overlap": CHUNK_OVERLAP,
    "strategies": list(chunkers.keys()),
}

with open("experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved all experiment results.")

In [ ]:
#from google.colab import files

#files.download("chunking_summary.csv")

## Retrieval experiments: dense vs. BM25 vs. hybrid (RRF)

Using below mathods:

- **dense**: all-MiniLM-L6-v2 + FAISS (built above btw)
- **bm25**: keyword ranking (`rank_bm25`)
- **hybrid**: Reciprocal Rank Fusion of the dense and BM25 rankings

Metrics: Recall@K, Precision@K, MRR, nDCG@K, F1@K, plus per-query retrieval latency.

In [ ]:
RETRIEVAL_METHODS = ["dense", "bm25", "hybrid"]

RRF_K = 60  #standard RRF constant (Cormack et al. 2009)
CANDIDATE_POOL = 50   # how deep each retriever's ranking goes before fusion

print("Methods:", RETRIEVAL_METHODS)
print("RRF k:", RRF_K, "| candidate pool:", CANDIDATE_POOL)

In [ ]:
from rank_bm25 import BM25Okapi

STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "for", "to", "and", "or", "is", "are",
    "was", "were", "be", "been", "do", "does", "did", "they", "their", "it",
    "its", "this", "that", "these", "those", "with", "by", "as", "at", "from",
    "what", "which", "who", "how", "we", "our", "can", "has", "have", "had",
}


def tokenize(text):
    tokens = re.findall(r"\w+", text.lower())
    return [t for t in tokens if t not in STOPWORDS]


def add_bm25_indexes(indexes_by_strategy):
    for strategy, paper_indexes in indexes_by_strategy.items():
        for paper_id, data in paper_indexes.items():
            corpus = [
                tokenize(f"{chunk['section']} {chunk['text']}")
                for chunk in data["chunks"]
            ]
            data["bm25"] = BM25Okapi(corpus)


add_bm25_indexes(indexes_by_strategy)
print("BM25 indexes ready.")

In [ ]:
# Encode every question once n also store encode time so dense/hybrid latency is fair.
query_embeddings = {}
query_encode_ms = {}

for q in evaluation_questions:
    text = q["question"]

    if text in query_embeddings:
        continue

    start = time.perf_counter()
    emb = model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    query_encode_ms[text] = (time.perf_counter() - start) * 1000
    query_embeddings[text] = emb

print("Encoded", len(query_embeddings), "unique questions.")

In [ ]:
def dense_ranking(question, data, n):
    index = data["index"]
    n = min(n, index.ntotal)

    scores, indices = index.search(query_embeddings[question], n)

    return [
        (int(i), float(s))
        for s, i in zip(scores[0], indices[0])
        if i >= 0
    ]


def bm25_ranking(question, data, n):
    scores = data["bm25"].get_scores(tokenize(question))
    order = np.argsort(-scores, kind="stable")[:n]

    return [(int(i), float(scores[i])) for i in order]


def rrf_fuse(rankings, k=RRF_K):
    #  score(d) = sum over rankers of 1 / (k + rank_of_d)
    fused = defaultdict(float)

    for ranking in rankings:
        for rank, (chunk_idx, _) in enumerate(ranking, start=1):
            fused[chunk_idx] += 1.0 / (k + rank)

    return sorted(fused.items(), key=lambda x: -x[1])


def rank_chunks(question, paper_id, paper_indexes, method, top_k=TOP_K):
    data = paper_indexes.get(paper_id)

    if data is None or data["index"].ntotal == 0:
        return []

    pool = max(top_k, CANDIDATE_POOL)

    if method == "dense":
        ranked = dense_ranking(question, data, pool)
    elif method == "bm25":
        ranked = bm25_ranking(question, data, pool)
    elif method == "hybrid":
        ranked = rrf_fuse([
            dense_ranking(question, data, pool),
            bm25_ranking(question, data, pool),
        ])
    else:
        raise ValueError(f"Unknown method: {method}")

    return ranked[:top_k]


def retrieve(question, paper_id, paper_indexes, method, top_k=TOP_K):
    data = paper_indexes.get(paper_id)

    return [
        {**data["chunks"][i], "score": s}
        for i, s in rank_chunks(question, paper_id, paper_indexes, method, top_k)
    ]

In [ ]:
def compute_metrics(relevance, total_relevant, k=TOP_K):
    rel = np.array(relevance[:k], dtype=float)
    hits = rel.sum()

    recall = hits / total_relevant
    precision = hits / k

    hit_ranks = np.flatnonzero(rel) + 1
    mrr = 1.0 / hit_ranks[0] if len(hit_ranks) else 0.0

    dcg = (rel / np.log2(np.arange(2, len(rel) + 2))).sum()
    ideal_n = min(total_relevant, k)
    idcg = (1.0 / np.log2(np.arange(2, ideal_n + 2))).sum()
    ndcg = dcg / idcg if idcg > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return {
        "recall_at_k": recall,
        "precision_at_k": precision,
        "mrr": mrr,
        "ndcg_at_k": ndcg,
        "f1_at_k": f1,
    }


# rlevance of every chunk for every question doesn't depend on the retriever, so compute it once per (strategy, question).
relevance_cache = {}

for strategy, paper_indexes in indexes_by_strategy.items():
    for q in evaluation_questions:
        data = paper_indexes.get(q["paper_id"])

        if data is None:
            continue

        relevance_cache[(strategy, q["question_id"])] = [
            chunk_matches_evidence(chunk["text"], q["evidence"])
            for chunk in data["chunks"]
        ]

print("Cached relevance labels:", len(relevance_cache))

In [ ]:
CROSS_ENCODER_MODEL = "cross-encoder/NV-RerankQA-Mistral-4B-v3"
class CrossEncoderReRanker:
    def __init__(self, model_name=CROSS_ENCODER_MODEL):
        self.model_name = model_name
        self.model = CrossEncoder(model_name)

    def rerank(self, question, chunks):
        if not chunks:
            return []

        pairs = [
            (question, chunk["text"])
            for chunk in chunks
        ]
        scores = self.model.predict(pairs)
        return _apply_scores(chunks, scores)


cross_encoder_reranker = CrossEncoderReRanker()
print("Cross-Encoder loaded:", CROSS_ENCODER_MODEL)

In [ ]:
class JinaReRanker:
    # reranker24: sends the question and all 20 chunks to Jina's reranking API in ONE call

    name = "jina"

    def __init__(self, model_name: str = "jina-reranker-v3"):
        self.model_name = model_name
        self.headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ['JINA_API_KEY']}" 
        }
    
def __call__(self, query: str, chunks: list[dict]) -> list[dict]:
        texts = [chunk["text"] for chunk in chunks]
        # takes just the text out of each of the 20 chunks

        response = requests.post(
            "https://api.jina.ai/v1/rerank",
            headers = self.headers,
            json={"model": self.model_name, "query": query, "documents": texts, "top_n": len(texts), "return_documents": False},
            # top_n = 20: ask Jina to score and return ALL 20 chunks, not just the best one
            
            timeout=60,
        )
        # ONE call to Jina: the question + 20 texts in
        # return_documents:False: we already have the texts, so Jina only needs to send back scores

        response.raise_for_status()    # if Jina replies with an error stop here and show it.

        scores = [0.0] * len(chunks)   # makes one empty slot per chunk, in the same order as chunks
                                       #     scores = [0.0, 0.0, 0.0]

        for result in response.json()["results"]:
            # response.json() turns Jina's reply into Python dictionaries and lists
            # ["results"] picks out the lists of results
            # the loop takes one result at a time, e.g. {"index": 2, "relevance_score": 0.91}


            #{"results": [ {"index": 2, "relevance_score": 0.91},
            #              {"index": 0, "relevance_score": 0.45},
            #              {"index": 1, "relevance_score": 0.12} ]}

            scores[result["index"]] = result["relevance_score"]
             # result["index"]           → which chunk this is about, e.g. 2 (the 3rd chunk)
             # result["relevance_score"] → its score, e.g. 0.91
             # puts the score into that chunk's slot:
             #     after result 1:  [0.0,  0.0,  0.91]
             #     after result 2:  [0.45, 0.0,  0.91]
             #     after result 3:  [0.45, 0.12, 0.91]
             

In [ ]:
from rag.chat_model import EuriChatModel


class LLMReRanker:
    def __init__(self):
        self.model = EuriChatModel()

    def rerank(self, question, chunks):
        if not chunks:
            return []

        numbered_chunks = "\n\n".join(
            f"CHUNK {i}: {chunk['text']}"
            for i, chunk in enumerate(chunks, start=1)
        )

        prompt = f"""
You are a retrieval reranking system.
Rank the following chunks by how well they answer the question.
Use only the information in the chunks.

QUESTION:
{question}

CHUNKS:
{numbered_chunks}

Return ONLY the chunk numbers in best-to-worst order, separated by commas.
Include each chunk number exactly once.
"""

        response = self.model.invoke(prompt)
        response_text = (
            response.content
            if hasattr(response, "content")
            else str(response)
        )

        ranked_ids = [
            int(value)
            for value in re.findall(r"\b\d+\b", response_text)
            if 1 <= int(value) <= len(chunks)
        ]

        # Remove duplicates while preserving the LLM's ordering.
        ranked_ids = list(dict.fromkeys(ranked_ids))

        # Add any omitted chunks at the end in their original retrieval order.
        ranked_ids.extend(
            i for i in range(1, len(chunks) + 1)
            if i not in ranked_ids
        )

        reordered = [dict(chunks[i - 1]) for i in ranked_ids]
        scores = list(
            range(len(reordered), 0, -1)
        )

        return _apply_scores(reordered, scores)


llm_reranker = LLMReRanker()
print("LLM reranker initialized.")

In [ ]:
def evaluate_config(strategy, method, paper_indexes):
    rows = []

    for q in evaluation_questions:
        flags = relevance_cache.get((strategy, q["question_id"]))

        if flags is None:
            continue

        total_relevant = sum(flags)

        if total_relevant == 0:
            rows.append({
                "strategy": strategy,
                "method": method,
                "question_id": q["question_id"],
                "evaluated": False,
            })
            continue

        start = time.perf_counter()
        ranked = rank_chunks(
            q["question"], q["paper_id"], paper_indexes, method, TOP_K,
        )
        latency_ms = (time.perf_counter() - start) * 1000

        if method in ("dense", "hybrid"):
            latency_ms += query_encode_ms[q["question"]]

        relevance = [flags[i] for i, _ in ranked]

        rows.append({
            "strategy": strategy,
            "method": method,
            "question_id": q["question_id"],
            "evaluated": True,
            "n_relevant_in_paper": total_relevant,
            "latency_ms": latency_ms,
            **compute_metrics(relevance, total_relevant, TOP_K),
        })

    return pd.DataFrame(rows)

In [ ]:
grid_frames = []

for strategy, paper_indexes in indexes_by_strategy.items():
    for method in RETRIEVAL_METHODS:
        print(f"Evaluating: {strategy} + {method}")
        grid_frames.append(evaluate_config(strategy, method, paper_indexes))

grid_df = pd.concat(grid_frames, ignore_index=True)

# Different chunkers can leave different questions "evaluable" (evidence has to
# match at least one chunk). Compare all 9 configs on the SAME questions.
evaluated_sets = [
    set(grid_df[(grid_df["strategy"] == s) & grid_df["evaluated"]]["question_id"])
    for s in indexes_by_strategy
]
common_ids = set.intersection(*evaluated_sets)

grid_common = grid_df[
    grid_df["evaluated"] & grid_df["question_id"].isin(common_ids)
]

print(f"\nQuestions evaluable under every chunker: {len(common_ids)}")

In [ ]:
metric_cols = [
    "recall_at_k", "precision_at_k", "mrr", "ndcg_at_k", "f1_at_k",
]

grid_summary = (
    grid_common
    .groupby(["strategy", "method"])
    .agg(
        n_questions=("question_id", "count"),
        recall_at_k=("recall_at_k", "mean"),
        precision_at_k=("precision_at_k", "mean"),
        mrr=("mrr", "mean"),
        ndcg_at_k=("ndcg_at_k", "mean"),
        f1_at_k=("f1_at_k", "mean"),
        latency_ms=("latency_ms", "mean"),
        recall_se=("recall_at_k", "sem"),
        mrr_se=("mrr", "sem"),
    )
    .reset_index()
    .sort_values("ndcg_at_k", ascending=False)
)

display(grid_summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, metric in zip(axes, ["recall_at_k", "mrr", "ndcg_at_k"]):
    pivot = grid_summary.pivot(index="strategy", columns="method", values=metric)
    pivot = pivot[RETRIEVAL_METHODS]

    im = ax.imshow(pivot.values, cmap="Blues", vmin=0, vmax=max(0.6, pivot.values.max()))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f"{metric} (top-{TOP_K})")

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center")

plt.tight_layout()
plt.show()

grid_summary.pivot(index="strategy", columns="method", values=["recall_at_k", "mrr", "ndcg_at_k"])[
    "recall_at_k"
][RETRIEVAL_METHODS].plot(
    kind="bar", figsize=(8, 4), ylim=(0, 1),
    title=f"Recall@{TOP_K} by chunking x retrieval", rot=0,
)
plt.ylabel("Recall")
plt.tight_layout()
plt.show()

In [ ]:
# How often does each retriever find a relevant chunk in the top-K when the other doesnt?
# Note --> this is the case for hybrid. it only helps if dense and BM25 miss different questions
rows = []

for strategy in indexes_by_strategy:
    sub = grid_common[grid_common["strategy"] == strategy]
    hit = (
        sub.assign(hit=sub["recall_at_k"] > 0)
        .pivot(index="question_id", columns="method", values="hit")
    )

    rows.append({
        "strategy": strategy,
        "both_hit": (hit["dense"] & hit["bm25"]).mean(),
        "dense_only": (hit["dense"] & ~hit["bm25"]).mean(),
        "bm25_only": (~hit["dense"] & hit["bm25"]).mean(),
        "neither": (~hit["dense"] & ~hit["bm25"]).mean(),
        "hybrid_hit": hit["hybrid"].mean(),
    })

display(pd.DataFrame(rows).round(3))

In [ ]:
# Same example question as above, now for every retrieval method (section_aware chunks).
example_strategy = "section_aware"

for method in RETRIEVAL_METHODS:
    print("\n" + "=" * 80)
    print("METHOD:", method)

    retrieved = retrieve(
        example["question"],
        example["paper_id"],
        indexes_by_strategy[example_strategy],
        method,
        TOP_K,
    )

    for rank, chunk in enumerate(retrieved, start=1):
        relevant = chunk_matches_evidence(chunk["text"], example["evidence"])
        print(f"Rank {rank} | score {chunk['score']:.3f} | match: {relevant} | {chunk['section']}")

In [ ]:
grid_summary.to_csv("retrieval_grid_summary.csv", index=False)
grid_df.to_csv("retrieval_grid_question_results.csv", index=False)

retrieval_config = {
    **config,
    "retrieval_methods": RETRIEVAL_METHODS,
    "rrf_k": RRF_K,
    "candidate_pool": CANDIDATE_POOL,
    "bm25": "rank_bm25.BM25Okapi (default k1/b), stopwords removed",
    "common_questions_evaluated": len(common_ids),
}

with open("experiment_config_retrieval.json", "w") as f:
    json.dump(retrieval_config, f, indent=2)

print("Saved retrieval results.")

In [ ]:
from google.colab import files

files.download("retrieval_grid_summary.csv")